# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

In [13]:
'''
Hint: beware of missing rows of data.
The source is missing a few months!
'''

'\nHint: beware of missing rows of data.\nThe source is missing a few months!\n'

In [14]:
# YOUR CHANGES HERE

# ======================
# Important Libraries 
# ======================
%pip install numpy 
%pip install pandas
%pip install matplotlib
%pip install statsmodels
%pip install prophet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.tsa.ar_model as ar_model
import statsmodels.graphics.tsaplots as tsaplots
from prophet import Prophet

# ====================================
# Import Strawberry Prices dataset 
# ====================================
df = pd.read_csv('strawberry-prices.tsv', sep='\t')

# Inspect dataset 
# df.info()  # summary
# df.describe()  # summary statistics

# Show missing and N/A values
# df.isna().sum()  # count number of missing values
# df.head(10)


# =========================================================================
# Model 1 (Simple): Use years 2020-2024 to predict monthly prices in 2025
# =========================================================================
# X = df[(df['month'] >= '2020-01-01') & (df['month'] <= '2024-12-31')]
# print(X)
# y = df[(df['month'] >= '2025-01-01') & (df['month'] <= '2025-12-01')]['price']
# print(y)

# strawberry_model = Prophet()
# strawberry_model.fit(X.rename(columns={'month': 'ds', 'price': 'y'}))

# Make future dataframe for 2025
# strawberry_future = strawberry_model.make_future_dataframe(periods=12, freq='ME')
# strawberry_forecast = strawberry_model.predict(strawberry_future)
# strawberry_forecast


# =================================================================
# Model 2: Use years 2020-2024 to predict monthly prices in 2025
# =================================================================
df = pd.read_csv('strawberry-prices.tsv', sep='\t', parse_dates=['month'])
df = df.sort_values('month')
df['month_number'] = df['month'].dt.month

# Keep observed values only
observed = df[(df['month'] >= '2020-01-01') & (df['month'] <= '2025-12-01')].copy()

def prophet_forecast(train, forecast_months):
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
    )
    model.fit(train[['month', 'price']].rename(columns={'month': 'ds', 'price': 'y'}))
    prediction = model.predict(pd.DataFrame({'ds': pd.to_datetime(forecast_months)}))
    return prediction['yhat'].to_numpy()


def seasonal_mean_forecast(train, forecast_months):
    month_means = train.groupby(train['month'].dt.month)['price'].mean()
    overall_mean = train['price'].mean()
    return np.array([month_means.get(month.month, overall_mean) for month in forecast_months])


def seasonal_last_forecast(train, forecast_months):
    result = []
    for month in forecast_months:
        history = train[train['month'].dt.month == month.month]
        result.append(history.iloc[-1]['price'] if not history.empty else train['price'].mean())
    return np.array(result)

backtest_rows = []
for target_year in [2020, 2021, 2022, 2023, 2024]:
    train = observed[observed['month'].dt.year < target_year]
    target = observed[observed['month'].dt.year == target_year]
    target_months = target['month'].tolist()
    # Backtest every requested year that has both historical training data
    # and observed target values. Years without prior history are skipped.
    train = train.dropna(subset=['month', 'price']).copy()
    target = target.dropna(subset=['month', 'price']).copy()

    if train.empty or target.empty or len(train) < 2:
        continue

    # Add missing calendar months to the training data and replace prices with the mean price for corresponding year
    def fill_missing_months_with_year_mean(data):
        data = data[['month', 'price']].copy()
        data['month'] = pd.to_datetime(data['month'])
        data = data.sort_values('month')

        full_months = pd.date_range(
            data['month'].min().replace(day=1),
            data['month'].max().replace(day=1),
            freq='MS',
        )

        completed = (
            pd.DataFrame({'month': full_months})
            .merge(data, on='month', how='left')
            .sort_values('month')
        )

        completed['year'] = completed['month'].dt.year
        year_means = completed.groupby('year')['price'].transform('mean')
        overall_mean = completed['price'].mean()

        completed['price'] = (
            completed['price']
            .fillna(year_means)
            .fillna(overall_mean)
        )

        return completed.drop(columns='year')

    train = fill_missing_months_with_year_mean(train)
    target_months = target['month'].tolist()

    forecasts = {
        'prophet': prophet_forecast(train, target_months),
        'seasonal_mean': seasonal_mean_forecast(train, target_months),
        'seasonal_last': seasonal_last_forecast(train, target_months),
    }
    for method, prediction in forecasts.items():
        residual = target['price'].to_numpy() - prediction
        backtest_rows.append({
            'method': method,
            'year': target_year,
            'rmse': np.sqrt(np.mean(residual ** 2)),
            'mae': np.mean(np.abs(residual)),
        })

# Metrics
model_scores = pd.DataFrame(backtest_rows).groupby('method')[['rmse', 'mae']].mean().sort_values('rmse')
best_method = model_scores.index[0]
print('Rolling backtest scores:')
print(model_scores)
print(f'\nSelected method: {best_method}')

# Fit selected method on all observations from 2020-2024; forecast all 12 months in 2025
training = observed[observed['month'] < '2025-01-01'].copy()
forecast_months = pd.date_range('2025-01-01', '2025-12-01', freq='MS')
if best_method == 'prophet':
    prediction = prophet_forecast(training, forecast_months)
elif best_method == 'seasonal_mean':
    prediction = seasonal_mean_forecast(training, forecast_months)
else:
    prediction = seasonal_last_forecast(training, forecast_months)

strawberry_backtest = pd.DataFrame({'month': forecast_months, 'price': prediction})
strawberry_backtest['month'] = strawberry_backtest['month'].dt.strftime('%Y-%m-01')


# ============================================================
# Save backtest to TSV file named 'strawberry-backtest.tsv'
# ============================================================
strawberry_backtest.to_csv('strawberry-backtest.tsv', sep='\t', index=False)
strawberry_backtest

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.


Note: you may need to restart the kernel to use updated packages.


02:49:07 - cmdstanpy - INFO - Chain [1] start processing
02:49:07 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
02:49:07 - cmdstanpy - INFO - Chain [1] start processing
02:49:16 - cmdstanpy - INFO - Chain [1] done processing
02:49:17 - cmdstanpy - INFO - Chain [1] start processing
02:49:17 - cmdstanpy - INFO - Chain [1] done processing
02:49:17 - cmdstanpy - INFO - Chain [1] start processing
02:49:17 - cmdstanpy - INFO - Chain [1] done processing


Rolling backtest scores:
                   rmse       mae
method                           
seasonal_mean  0.351640  0.305634
seasonal_last  0.361392  0.293516
prophet        1.598091  1.374359

Selected method: seasonal_mean


,month,price
0,2025-01-01,4.5012
1,2025-02-01,4.1256
2,2025-03-01,3.6994
3,2025-04-01,3.8730
4,2025-05-01,3.4684
5,2025-06-01,3.2154
6,2025-07-01,3.1784
7,2025-08-01,3.4612
8,2025-09-01,3.6138
9,2025-10-01,3.8842


Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [15]:
# YOUR CHANGES HERE

# =============================================================================================
# Calculate mean and std of residuals between backtest predictions and actual observed prices
# =============================================================================================
# Ensure both month columns have same data type
strawberry_backtest['month'] = pd.to_datetime(strawberry_backtest['month'])
observed['month'] = pd.to_datetime(observed['month'])

merged = observed[['month', 'price']].merge(
    strawberry_backtest.rename(columns={'price': 'predicted_price'}),
    on='month',
    how='inner'
)

residuals = merged['price'] - merged['predicted_price']

residuals_mean = residuals.mean()
residuals_std = residuals.std()

print(f'Mean of residuals: {residuals_mean}')
print(f'STD of residuals: {residuals_std}')


# =========================================================================
# Write mean and std of residuals to csv file named backtest-accuracy.tsv
# =========================================================================
backtest_accuracy_df = pd.DataFrame({
    'mean': [residuals_mean],
    'std': [residuals_std]
})

# Save file
backtest_accuracy_df.to_csv('backtest-accuracy.tsv', sep='\t', index=False)
backtest_accuracy_df

Mean of residuals: -0.06554000000000001
STD of residuals: 0.1531480053920245


,mean,std
0,-0.06554,0.153148


Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [16]:
# YOUR CHANGES HERE

# ========================================================
# Use years 2020-2025 to predict monthly prices in 2026
# ========================================================
df = pd.read_csv('strawberry-prices.tsv', sep='\t', parse_dates=['month'])
df = df.sort_values('month')
df['month_number'] = df['month'].dt.month

# Keep observed values only
observed = df[(df['month'] >= '2020-01-01') & (df['month'] <= '2025-12-01')].copy()

def prophet_forecast(train, forecast_months):
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
    )
    model.fit(train[['month', 'price']].rename(columns={'month': 'ds', 'price': 'y'}))
    prediction = model.predict(pd.DataFrame({'ds': pd.to_datetime(forecast_months)}))
    return prediction['yhat'].to_numpy()


def seasonal_mean_forecast(train, forecast_months):
    month_means = train.groupby(train['month'].dt.month)['price'].mean()
    overall_mean = train['price'].mean()
    return np.array([month_means.get(month.month, overall_mean) for month in forecast_months])


def seasonal_last_forecast(train, forecast_months):
    result = []
    for month in forecast_months:
        history = train[train['month'].dt.month == month.month]
        result.append(history.iloc[-1]['price'] if not history.empty else train['price'].mean())
    return np.array(result)

backtest_rows = []
for target_year in [2020, 2021, 2022, 2023, 2024, 2025]:
    train = observed[observed['month'].dt.year < target_year]
    target = observed[observed['month'].dt.year == target_year]
    target_months = target['month'].tolist()
    # Backtest every requested year that has both historical training data
    # and observed target values. Years without prior history are skipped.
    train = train.dropna(subset=['month', 'price']).copy()
    target = target.dropna(subset=['month', 'price']).copy()

    if train.empty or target.empty or len(train) < 2:
        continue

    # Add missing calendar months to the training data and replace prices with the mean price for corresponding year
    def fill_missing_months_with_year_mean(data):
        data = data[['month', 'price']].copy()
        data['month'] = pd.to_datetime(data['month'])
        data = data.sort_values('month')

        full_months = pd.date_range(
            data['month'].min().replace(day=1),
            data['month'].max().replace(day=1),
            freq='MS',
        )

        completed = (
            pd.DataFrame({'month': full_months})
            .merge(data, on='month', how='left')
            .sort_values('month')
        )

        completed['year'] = completed['month'].dt.year
        year_means = completed.groupby('year')['price'].transform('mean')
        overall_mean = completed['price'].mean()

        completed['price'] = (
            completed['price']
            .fillna(year_means)
            .fillna(overall_mean)
        )

        return completed.drop(columns='year')

    train = fill_missing_months_with_year_mean(train)
    target_months = target['month'].tolist()

    forecasts = {
        'prophet': prophet_forecast(train, target_months),
        'seasonal_mean': seasonal_mean_forecast(train, target_months),
        'seasonal_last': seasonal_last_forecast(train, target_months),
    }
    for method, prediction in forecasts.items():
        residual = target['price'].to_numpy() - prediction
        backtest_rows.append({
            'method': method,
            'year': target_year,
            'rmse': np.sqrt(np.mean(residual ** 2)),
            'mae': np.mean(np.abs(residual)),
        })

# Metrics
model_scores = pd.DataFrame(backtest_rows).groupby('method')[['rmse', 'mae']].mean().sort_values('rmse')
best_method = model_scores.index[0]
print('Rolling backtest scores:')
print(model_scores)
print(f'\nSelected method: {best_method}')

# Fit selected method on all observations from 2020-2025; forecast all 12 months in 2026
training = observed[observed['month'] < '2026-01-01'].copy()
forecast_months = pd.date_range('2026-01-01', '2026-12-01', freq='MS')
if best_method == 'prophet':
    prediction = prophet_forecast(training, forecast_months)
elif best_method == 'seasonal_mean':
    prediction = seasonal_mean_forecast(training, forecast_months)
else:
    prediction = seasonal_last_forecast(training, forecast_months)

strawberry_forecast = pd.DataFrame({'month': forecast_months, 'price': prediction})
strawberry_forecast['month'] = strawberry_forecast['month'].dt.strftime('%Y-%m-01')


# ============================================================
# Save forecast to TSV file named 'strawberry-forecast.tsv'
# ============================================================
strawberry_forecast.to_csv('strawberry-forecast.tsv', sep='\t', index=False)
strawberry_forecast

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
02:49:17 - cmdstanpy - INFO - Chain [1] start processing


02:49:17 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
02:49:17 - cmdstanpy - INFO - Chain [1] start processing
02:49:27 - cmdstanpy - INFO - Chain [1] done processing
02:49:27 - cmdstanpy - INFO - Chain [1] start processing
02:49:27 - cmdstanpy - INFO - Chain [1] done processing
02:49:27 - cmdstanpy - INFO - Chain [1] start processing
02:49:28 - cmdstanpy - INFO - Chain [1] done processing
02:49:28 - cmdstanpy - INFO - Chain [1] start processing
02:49:28 - cmdstanpy - INFO - Chain [1] done processing


Rolling backtest scores:
                   rmse       mae
method                           
seasonal_mean  0.315681  0.268432
seasonal_last  0.334522  0.272093
prophet        1.324526  1.136727

Selected method: seasonal_mean


,month,price
0,2026-01-01,4.515000
1,2026-02-01,4.117500
2,2026-03-01,3.644333
3,2026-04-01,3.802000
4,2026-05-01,3.459667
5,2026-06-01,3.211167
6,2026-07-01,3.179833
7,2026-08-01,3.456000
8,2026-09-01,3.619000
9,2026-10-01,3.884200


Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [17]:
# YOUR CHANGES HERE

# ================================================================
# Use 2026 forecast to analyze profit 
# (pick different pairs of months to buy and sell strawberries)
# ================================================================
from itertools import combinations

investment = 1000000
freeze_cost = 0.20      # per pint
storage_cost = 0.10     # monthly
selling_discount = 0.90     #10% discount

# Use 2026 forecast
strawberry_forecast = pd.read_csv("strawberry-forecast.tsv", sep="\t", parse_dates=["month"])
strawberry_forecast["price"] = pd.to_numeric(strawberry_forecast["price"], errors="coerce")
strawberry_forecast = strawberry_forecast.dropna(subset=["month", "price"]).reset_index(drop=True)

# Analyze all possible buy/sell month combos
timing_rows = []

for buy_row, sell_row in combinations(strawberry_forecast.itertuples(index=False), 2):
    buy_month = buy_row.month
    sell_month = sell_row.month
    buy_price = float(buy_row.price)
    sell_price = float(sell_row.price)

    # number months stored
    months_stored = ((sell_month.year - buy_month.year) * 12 + sell_month.month - buy_month.month)

    # total cost per pint (incl freezing + storage)
    cost_per_pint = (buy_price + freeze_cost + storage_cost * months_stored)

    # Purchase as many pints as possible with$1,000,000 
    pints_purchased = int(np.floor(investment / cost_per_pint))

    # calculate expected profit per pint
    expected_profit_per_pint = (sell_price * selling_discount - cost_per_pint)
    expected_profit = pints_purchased * expected_profit_per_pint

    timing_rows.append({
        "buy_month": buy_month.strftime("%Y-%m-01"),
        "sell_month": sell_month.strftime("%Y-%m-01"),
        "pints_purchased": pints_purchased,
        "expected_profit": expected_profit,
    })

timings = pd.DataFrame(timing_rows)


# ================================================
# Save analysis to TSV file named 'timings.tsv'
# ================================================
timings.to_csv("timings.tsv", sep="\t", index=False)
timings

,buy_month,sell_month,pints_purchased,expected_profit
0,2026-01-01,2026-02-01,207684,-230373.47700
1,2026-01-01,2026-03-01,203458,-332674.17580
2,2026-01-01,2026-04-01,199401,-317685.67320
3,2026-01-01,2026-05-01,195503,-391260.15390
4,2026-01-01,2026-06-01,191754,-445818.46230
...,...,...,...,...
61,2026-09-01,2026-11-01,248818,-14187.60236
62,2026-09-01,2026-12-01,242777,78514.08180
63,2026-10-01,2026-11-01,238994,-53109.24668
64,2026-10-01,2026-12-01,233415,36926.25300


Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [19]:
# YOUR CHANGES HERE

# ====================================================================
# Select best profit scenario according to previous timing analysis
# ====================================================================
best_profit = timings.sort_values("expected_profit", ascending=False).head(1)
best_profit


# ========================================================================================
# Calculate how much profit changes if sell price is off by 1 STD from backtest analysis
# ========================================================================================
backtest_std = strawberry_backtest["price"].std()
pints_purchased = best_profit["pints_purchased"].iloc[0]
one_std_profit = pints_purchased * backtest_std


# ================================================
# Save results to TSV file named 'check.tsv'
# ================================================
check = pd.DataFrame({
    "best_profit": [best_profit["expected_profit"].iloc[0]],
    "one_std_profit": [one_std_profit]
})

# Save file
check.to_csv("check.tsv", sep="\t", index=False)
check

,best_profit,one_std_profit
0,144997.620367,138694.744038


Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.